# 🛡️ Kaggle 24/7 Full Web Platform + Self-Hosted Qwen3.6-12B GGUF Engine
Téléchargement direct du dépôt HuggingFace https://huggingface.co/KevinJK51/Qwen3.6-12B-IQ-Ultra-Heretic-Uncensored-Thinking-V2-Hightop-GGUF, 5 optimisations matérielles et tunnel HTTPS.

In [ ]:
# 1. Test de connectivité réseau HTTP 200
import urllib.request

print('🔍 Vérification de la connectivité réseau Kaggle...')
try:
    req = urllib.request.urlopen('https://httpbin.org/status/200', timeout=5)
    if req.getcode() == 200:
        print('🟢 RÉSEAU OK : Connexion Internet établie (Code HTTP 200)')
except Exception as e:
    print(f'⚠️ NOTE RÉSEAU : {e}')

In [ ]:
# 2. Clonage résilient du dépôt Git OSINT
import time, os, subprocess

!rm -rf /kaggle/working/projet_osint

clone_url = 'https://github.com/your-repo/projet_osint.git'
cloned = False

for attempt in range(1, 6):
    print(f'Tentative de clonage Git ({attempt}/5)...')
    res = subprocess.run(['git', 'clone', clone_url, '/kaggle/working/projet_osint'])
    if res.returncode == 0:
        cloned = True
        print('🟢 Dépôt Git OSINT cloné avec succès !')
        break
    time.sleep(3)

if not cloned:
    raise RuntimeError('Échec du clonage Git OSINT.')

!pip install --no-cache-dir huggingface_hub "llama-cpp-python[server]"

In [ ]:
# 3. Clonage du modèle HuggingFace et Démarrage de llama-server
import os, subprocess, time, glob, shutil

hf_repo_url = 'https://huggingface.co/KevinJK51/Qwen3.6-12B-IQ-Ultra-Heretic-Uncensored-Thinking-V2-Hightop-GGUF'
model_dir = '/kaggle/working/models/qwen12b'
os.makedirs(model_dir, exist_ok=True)

print(f'⬇️ Clonage direct du dépôt modèle HuggingFace : {hf_repo_url}...')
!git clone {hf_repo_url} {model_dir} || true

gguf_files = glob.glob(f'{model_dir}/*.gguf')
if not gguf_files:
    from huggingface_hub import snapshot_download
    print('Utilisation du snapshot_download HuggingFace en fallback...')
    model_dir = snapshot_download(repo_id='KevinJK51/Qwen3.6-12B-IQ-Ultra-Heretic-Uncensored-Thinking-V2-Hightop-GGUF')
    gguf_files = glob.glob(f'{model_dir}/*.gguf')

if not gguf_files:
    raise RuntimeError('Aucun fichier .gguf trouvé dans le modèle téléchargé.')

model_path = gguf_files[0]
print(f'🟢 Modèle GGUF identifié et chargé : {model_path}')

print('🚀 Lancement de llama-server avec les 5 optimisations matérielles...')
llama_cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8080',
    '--n_ctx', '32768',
    '--n_threads', '2'
]
llama_process = subprocess.Popen(llama_cmd)
time.sleep(10)

In [ ]:
# 4. Démarrage de FastAPI & Tunnel HTTPS Cloudflare
%cd /kaggle/working/projet_osint/backend
!pip install --no-cache-dir -r requirements.txt
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared || true

print('Démarrage du Serveur FastAPI (Frontend + Backend) sur port 8000...')
server_process = subprocess.Popen(['python', '-m', 'app.main'])
time.sleep(6)

print('Lancement du Tunnel HTTPS...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(20):
    line = tunnel_process.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n======================================================')
            print(f'🚀 VOTRE INTERFACE WEB EST EN LIGNE H24 : {match.group(0)}')
            print('======================================================\n')
            break
    time.sleep(1)

In [ ]:
# 5. Boucle d'exécution continue 24/7 avec Checkpoints SQLite horaires
import time
from app.cloud_sync.kaggle_persistence import KagglePersistenceManager
from app.cloud_sync.garbage_collector import GarbageCollectorManager

print('🟢 Boucle d\'exécution continue 24/7 active...')
for hour in range(1, 11):
    time.sleep(3600)
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] Checkpoint State Heure {hour}/10...')
    GarbageCollectorManager.cleanup_temp_storage()
    KagglePersistenceManager.checkpoint_state()